## MLP: 노드 특징만 사용하는 비교군

GCNConv를 Linear로 교체하고 `34 → 4 → 4 → 2 → 4` 구조와 relu를 유지합니다. 

특징은 노드 ID를 나타내는 단위 행렬입니다. MLP의 첫 층에서 평가 노드에 대응하는 열은 학습 손실로부터 갱신되지 않으므로, 학습 정확도가 높아도 미관측 노드를 잘 분류하기 어렵습니다. 이 결과는 해당 특징과 레이블 수에서의 비교이며 MLP 전반의 성능을 뜻하지 않습니다.


## 1. Karate Club 데이터 불러오기

In [1]:
from torch_geometric.datasets import KarateClub

dataset = KarateClub()

data = dataset[0]
train_mask = data.train_mask.clone()
test_mask = ~train_mask

c:\Users\KETI\anaconda3\envs\ai_tutorial\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(dataset.num_features, 4)
        self.fc2 = nn.Linear(4, 4)
        self.fc3 = nn.Linear(4, 2)
        self.classifier = nn.Linear(2, dataset.num_classes)

    def forward(self, x, edge_index=None):
        # 공통 학습 함수와 인터페이스를 맞추며 edge_index는 사용하지 않습니다.
        h = self.fc1(x).relu()
        h = self.fc2(h).relu()
        h = self.fc3(h).relu()
        return self.classifier(h), h

model = MLP()

print(model)

MLP(
  (fc1): Linear(in_features=34, out_features=4, bias=True)
  (fc2): Linear(in_features=4, out_features=4, bias=True)
  (fc3): Linear(in_features=4, out_features=2, bias=True)
  (classifier): Linear(in_features=2, out_features=4, bias=True)
)


In [3]:
criterion = torch.nn.CrossEntropyLoss()  # Define loss criterion.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Define optimizer.

def train(data):
    optimizer.zero_grad()  # Clear gradients.
    out, h = model(data.x)  # Perform a single forward pass.
    loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Compute the loss solely based on the training nodes.
    loss.backward()  # Derive gradients.
    optimizer.step()  # Update parameters based on gradients.
    return loss, h

embedding_history = []
epoch_history = []
loss_history = []

for epoch in range(500):
    loss, h = train(data)
    if epoch % 1 == 0:
        # visualize_embedding(h, color=data.y, epoch=epoch, loss=loss)
        # time.sleep(0.3)
        embedding_history.append(h.detach().cpu().numpy().copy())
        epoch_history.append(epoch)
        loss_history.append(loss.item())

In [ ]:
model.eval()
logits, _ = model(data.x)
test_loss = F.cross_entropy(logits[test_mask], data.y[test_mask]).item()
test_accuracy = (
        F.softmax(logits[test_mask],dim=1).argmax(dim=1) == data.y[test_mask]
    ).float().mean().item()

print("test loss: ", test_loss)
print("test accuracy: ", test_accuracy)

test loss:  6.602901935577393
test accuracy:  0.36666667461395264
